In [1]:
import pandas as pd

# CSV-Datei einlesen
# Quelle: https://www.landesdatenbank.nrw.de/
df = pd.read_csv("../data/raw/2024/12411-09i.csv", sep=";", encoding="latin1", skiprows=6)

# Überblick:
print(df.head())
df.info()


   Unnamed: 0  Unnamed: 1                      Unnamed: 2 Insgesamt  \
0         NaN         NaN                             NaN    Anzahl   
1  31.12.2024         5.0             Nordrhein-Westfalen  18034454   
2  31.12.2024        51.0    Düsseldorf, Regierungsbezirk   5244379   
3  31.12.2024      5111.0         Düsseldorf, krfr. Stadt    618685   
4  31.12.2024      5112.0           Duisburg, krfr. Stadt    502270   

  unter 3 Jahre 3 bis unter 6 Jahre 6 bis unter 10 Jahre  \
0        Anzahl              Anzahl               Anzahl   
1        474056              520177               709988   
2        136019              150061               207321   
3         16116               15989                22216   
4         14347               15439                21192   

  10 bis unter 15 Jahre 15 bis unter 18 Jahre 18 bis unter 20 Jahre  ...  \
0                Anzahl                Anzahl                Anzahl  ...   
1                842053                517232               

In [2]:
#relevante Spalten filtern
df = df[['Unnamed: 2', 'Insgesamt', '6 bis unter 10 Jahre', '10 bis unter 15 Jahre', '15 bis unter 18 Jahre', '18 bis unter 20 Jahre']]
df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")
print(df)

                        Unnamed: 2 Insgesamt 6 bis unter 10 Jahre  \
0                              NaN    Anzahl               Anzahl   
1              Nordrhein-Westfalen  18034454               709988   
2     Düsseldorf, Regierungsbezirk   5244379               207321   
3          Düsseldorf, krfr. Stadt    618685                22216   
4            Duisburg, krfr. Stadt    502270                21192   
..                             ...       ...                  ...   
57                Märkischer Kreis    408046                15904   
58                     Olpe, Kreis    132346                 5195   
59      Siegen-Wittgenstein, Kreis    274074                10904   
60                    Soest, Kreis    300297                11631   
61                     Unna, Kreis    393784                15056   

   10 bis unter 15 Jahre 15 bis unter 18 Jahre 18 bis unter 20 Jahre  
0                 Anzahl                Anzahl                Anzahl  
1                 842053     

In [3]:
# Spalten umbenennen
df = df.rename(columns={"Unnamed: 2": "Name", "Insgesamt": "Gesamtbevölkerung"})

# Ints und Strings anpassen
df["Name"] = df["Name"].str.strip()
cols = df.columns.difference(["Name"])
df[cols] = (
     df[cols]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")  
)


# nach Kreisen und kreisfreien Städten filtern 
df = df[df["Name"].str.contains("Kreis|krfr. Stadt|Städteregion", case=False,na=False)]

In [4]:
# Altergruppen summieren
df["Bevölkerung 6 bis 20"] = df[[ '6 bis unter 10 Jahre', '10 bis unter 15 Jahre', '15 bis unter 18 Jahre', '18 bis unter 20 Jahre']].sum(axis=1)
df = df.drop(columns = ['6 bis unter 10 Jahre', '10 bis unter 15 Jahre', '15 bis unter 18 Jahre', '18 bis unter 20 Jahre'])
         

# Anteil berechnen
df["Anteil gesamt"] = (
    df["Bevölkerung 6 bis 20"] / df["Gesamtbevölkerung"] *100
)

In [5]:
print(df.head())
print(df[df["Gesamtbevölkerung"].isna()])
print(df[df['Anteil gesamt'].isna()])
print(df[df['Bevölkerung 6 bis 20'].isna()])

                           Name  Gesamtbevölkerung  Bevölkerung 6 bis 20  \
3       Düsseldorf, krfr. Stadt             618685                 76018   
4         Duisburg, krfr. Stadt             502270                 71210   
5            Essen, krfr. Stadt             574682                 76228   
6          Krefeld, krfr. Stadt             231406                 31097   
7  Mönchengladbach, krfr. Stadt             267213                 35894   

   Anteil gesamt  
3      12.287028  
4      14.177634  
5      13.264379  
6      13.438286  
7       13.43273  
                   Name  Gesamtbevölkerung  Bevölkerung 6 bis 20  \
19  Aachen, krfr. Stadt               <NA>                     0   
24        Aachen, Kreis               <NA>                     0   

    Anteil gesamt  
19           <NA>  
24           <NA>  
                   Name  Gesamtbevölkerung  Bevölkerung 6 bis 20  \
19  Aachen, krfr. Stadt               <NA>                     0   
24        Aachen, Kreis     

In [6]:
# Zeile löschen und Name anpassen
df = df[df["Name"] != "Essen, krfr. Stadt"]
df = df[df["Name"] != "Aachen, Kreis"]
df = df[df["Name"] != "Aachen, krfr. Stadt"]

print(df[df["Gesamtbevölkerung"].isna()])
print(df[df['Anteil gesamt'].isna()])
print(df[df['Bevölkerung 6 bis 20'].isna()])

Empty DataFrame
Columns: [Name, Gesamtbevölkerung, Bevölkerung 6 bis 20, Anteil gesamt]
Index: []
Empty DataFrame
Columns: [Name, Gesamtbevölkerung, Bevölkerung 6 bis 20, Anteil gesamt]
Index: []
Empty DataFrame
Columns: [Name, Gesamtbevölkerung, Bevölkerung 6 bis 20, Anteil gesamt]
Index: []


In [7]:
from ipynb.fs.defs.functions import clean_and_sort, validate_df

# Namen bereinigen
df = clean_and_sort(df, "Gesamtbevölkerung", "Bevölkerung 6 bis 20", "Anteil gesamt")
validate_df(df, 
    not_null=[ "Gesamtbevölkerung","Bevölkerung 6 bis 20", "Anteil gesamt"],
    positive=[ "Gesamtbevölkerung", "Bevölkerung 6 bis 20", "Anteil gesamt"],
    numeric=[ "Gesamtbevölkerung", "Bevölkerung 6 bis 20", "Anteil gesamt"],
    bounds={"Anteil gesamt": (0,100)},
    key_cols=["Name"])

DataFrame: alle Checks bestanden


[]

In [8]:
# speichern
df.to_csv("../data/processed/bevoelkerung_2024.csv", index=False)